# 02 — Data Cleaning

This notebook performs data cleaning and consistency validation on the
validated ICBHI respiratory sound dataset.

The raw ICBHI WAV and annotation files are kept unchanged.

Cleaning is performed on the intermediate metadata tables:

- patients.csv
- recordings.csv
- cycles.csv
- audio_info.csv

The cleaned outputs will be stored in data/processed/.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Interim data:", INTERIM_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Interim data: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim
Processed data: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed


In [2]:
patients_df = pd.read_csv(
    INTERIM_DATA_DIR / "patients.csv"
)

recordings_df = pd.read_csv(
    INTERIM_DATA_DIR / "recordings.csv"
)

cycles_df = pd.read_csv(
    INTERIM_DATA_DIR / "cycles.csv"
)

audio_info_df = pd.read_csv(
    INTERIM_DATA_DIR / "audio_info.csv"
)

print("Patients:", patients_df.shape)
print("Recordings:", recordings_df.shape)
print("Cycles:", cycles_df.shape)
print("Audio info:", audio_info_df.shape)

Patients: (126, 7)
Recordings: (920, 7)
Cycles: (6898, 12)
Audio info: (920, 4)


In [3]:
print("=== PATIENTS ===")
display(patients_df.isnull().sum())

print("\n=== RECORDINGS ===")
display(recordings_df.isnull().sum())

print("\n=== CYCLES ===")
display(cycles_df.isnull().sum())

print("\n=== AUDIO INFO ===")
display(audio_info_df.isnull().sum())

=== PATIENTS ===


patient_id       0
diagnosis        0
age              1
sex              1
adult_bmi       51
child_weight    82
child_height    84
dtype: int64


=== RECORDINGS ===


patient_id          0
recording_id        0
chest_location      0
acquisition_mode    0
equipment           0
wav_path            0
official_split      0
dtype: int64


=== CYCLES ===


patient_id          0
recording_id        0
chest_location      0
acquisition_mode    0
equipment           0
wav_path            0
cycle_number        0
start_time          0
end_time            0
crackles            0
wheezes             0
sound_label         0
dtype: int64


=== AUDIO INFO ===


file           0
sample_rate    0
frames         0
duration       0
dtype: int64

In [4]:
print("Duplicate patients:",
      patients_df["patient_id"].duplicated().sum())

print("Duplicate recordings:",
      recordings_df["wav_path"].duplicated().sum())

print("Duplicate cycles:",
      cycles_df.duplicated().sum())

Duplicate patients: 0
Duplicate recordings: 0
Duplicate cycles: 0


In [5]:
print("=== AGE GROUPS ===")

print("Age statistics:")
print(patients_df["age"].describe())

print("\nPatients with age < 18:")
display(
    patients_df[
        patients_df["age"].notna() &
        (patients_df["age"] < 18)
    ][
        [
            "patient_id",
            "diagnosis",
            "age",
            "sex",
            "adult_bmi",
            "child_weight",
            "child_height"
        ]
    ]
)

=== AGE GROUPS ===
Age statistics:
count    125.00000
mean      42.99264
std       32.20907
min        0.25000
25%        4.00000
50%       60.00000
75%       71.00000
max       93.00000
Name: age, dtype: float64

Patients with age < 18:


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
0,101,URTI,3.00,F,NaN,19.00,99.0
1,102,Healthy,0.75,F,NaN,9.80,73.0
4,105,URTI,7.00,F,NaN,32.00,135.0
7,108,LRTI,3.00,M,NaN,NaN,NaN
14,115,LRTI,0.58,M,NaN,7.14,64.0
18,119,URTI,2.00,F,NaN,15.20,94.0
20,121,Healthy,13.00,F,NaN,65.00,170.0
22,123,Healthy,5.00,M,NaN,25.00,125.0
24,125,Healthy,14.00,M,NaN,62.00,170.0
25,126,Healthy,1.00,F,NaN,10.18,80.0


In [6]:
print("Patients with age >= 18:")
display(
    patients_df[
        patients_df["age"].notna() &
        (patients_df["age"] >= 18)
    ][
        [
            "patient_id",
            "diagnosis",
            "age",
            "sex",
            "adult_bmi",
            "child_weight",
            "child_height"
        ]
    ].head(20)
)

Patients with age >= 18:


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
2,103,Asthma,70.0,F,33.00,NaN,NaN
3,104,COPD,70.0,F,28.47,NaN,NaN
5,106,COPD,73.0,F,21.00,NaN,NaN
6,107,COPD,75.0,F,33.70,NaN,NaN
8,109,COPD,84.0,F,33.53,NaN,NaN
9,110,COPD,75.0,M,25.21,NaN,NaN
10,111,Bronchiectasis,63.0,M,28.40,NaN,NaN
11,112,COPD,60.0,M,22.86,NaN,NaN
12,113,COPD,58.0,M,28.41,NaN,NaN
13,114,COPD,77.0,M,23.12,NaN,NaN


In [7]:
print("=== DEMOGRAPHIC MISSINGNESS BY AGE GROUP ===")

patients_check = patients_df.copy()

patients_check["age_group"] = np.where(
    patients_check["age"].isna(),
    "Unknown age",
    np.where(
        patients_check["age"] < 18,
        "Child (<18)",
        "Adult (>=18)"
    )
)

print("\nAge group counts:")
print(patients_check["age_group"].value_counts())

print("\nMissing values by age group:")
display(
    patients_check.groupby("age_group")[
        ["age", "sex", "adult_bmi", "child_weight", "child_height"]
    ].apply(lambda x: x.isna().sum())
)

=== DEMOGRAPHIC MISSINGNESS BY AGE GROUP ===

Age group counts:
age_group
Adult (>=18)    76
Child (<18)     49
Unknown age      1
Name: count, dtype: int64

Missing values by age group:


,age,sex,adult_bmi,child_weight,child_height
age_group,,,,,
Adult (>=18),0,0,1,76,76
Child (<18),0,0,49,5,7
Unknown age,1,1,1,1,1


In [8]:
print("=== PATIENTS WITH UNKNOWN AGE ===")

display(
    patients_df[
        patients_df["age"].isna()
    ]
)

=== PATIENTS WITH UNKNOWN AGE ===


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
122,223,COPD,NaN,NaN,NaN,NaN,NaN


In [9]:
print("=== PATIENT 223 DATA AVAILABILITY ===")

patient_id = 223

print("\nRecordings:")
display(
    recordings_df[
        recordings_df["patient_id"] == patient_id
    ]
)

print("\nNumber of recordings:",
      (recordings_df["patient_id"] == patient_id).sum())

print("\nRespiratory cycles:")
print(
    "Number of cycles:",
    (cycles_df["patient_id"] == patient_id).sum()
)

=== PATIENT 223 DATA AVAILABILITY ===

Recordings:


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,official_split
908,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
909,223,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
910,223,1b1,Ll,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
911,223,1b1,Lr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
912,223,1b1,Pl,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
913,223,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test



Number of recordings: 6

Respiratory cycles:
Number of cycles: 61


In [ ]:
Patient 223 has missing demographic information (age, sex, BMI,
weight, and height), but has valid diagnosis information and
valid respiratory-sound recordings/cycles. Therefore, the patient
is retained and demographic missing values are preserved as NaN.

In [10]:
print("=== CATEGORICAL VALUE CHECK ===")

print("\nDiagnoses:")
print(patients_df["diagnosis"].value_counts())

print("\nSex:")
print(patients_df["sex"].value_counts(dropna=False))

print("\nChest locations:")
print(recordings_df["chest_location"].value_counts())

print("\nAcquisition modes:")
print(recordings_df["acquisition_mode"].value_counts())

print("\nEquipment:")
print(recordings_df["equipment"].value_counts())

print("\nOfficial split:")
print(recordings_df["official_split"].value_counts())

=== CATEGORICAL VALUE CHECK ===

Diagnoses:
diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1
Name: count, dtype: int64

Sex:
sex
M      79
F      46
NaN     1
Name: count, dtype: int64

Chest locations:
chest_location
Ar    168
Al    162
Pl    139
Pr    132
Tc    130
Lr    112
Ll     77
Name: count, dtype: int64

Acquisition modes:
acquisition_mode
mc    732
sc    188
Name: count, dtype: int64

Equipment:
equipment
AKGC417L    646
Meditron    127
LittC2SE     87
Litt3200     60
Name: count, dtype: int64

Official split:
official_split
train    539
test     381
Name: count, dtype: int64


In [11]:
print("=== RESPIRATORY CYCLE LABEL CHECK ===")

print("\nCrackles values:")
print(cycles_df["crackles"].value_counts().sort_index())

print("\nWheezes values:")
print(cycles_df["wheezes"].value_counts().sort_index())

print("\nSound labels:")
print(cycles_df["sound_label"].value_counts())

print("\nUnique sound labels:")
print(cycles_df["sound_label"].unique())

=== RESPIRATORY CYCLE LABEL CHECK ===

Crackles values:
crackles
0    4528
1    2370
Name: count, dtype: int64

Wheezes values:
wheezes
0    5506
1    1392
Name: count, dtype: int64

Sound labels:
sound_label
Normal      3642
Crackles    1864
Wheezes      886
Both         506
Name: count, dtype: int64

Unique sound labels:
<StringArray>
['Normal', 'Wheezes', 'Crackles', 'Both']
Length: 4, dtype: str


In [12]:
print("=== RESPIRATORY CYCLE TIME VALIDATION ===")

cycles_check = cycles_df.copy()

cycles_check["cycle_duration"] = (
    cycles_check["end_time"] - cycles_check["start_time"]
)

print("Invalid start times:",
      (cycles_check["start_time"] < 0).sum())

print("Invalid end times:",
      (cycles_check["end_time"] <= cycles_check["start_time"]).sum())

print("Zero/negative cycle durations:",
      (cycles_check["cycle_duration"] <= 0).sum())

# Add recording duration
cycles_check = cycles_check.merge(
    audio_info_df[["file", "duration"]],
    left_on="wav_path",
    right_on="file",
    how="left"
)

print("Cycles with missing audio duration:",
      cycles_check["duration"].isna().sum())

print(
    "Cycles extending beyond recording:",
    (
        cycles_check["end_time"] >
        cycles_check["duration"]
    ).sum()
)

=== RESPIRATORY CYCLE TIME VALIDATION ===
Invalid start times: 0
Invalid end times: 0
Zero/negative cycle durations: 0
Cycles with missing audio duration: 6898
Cycles extending beyond recording: 0


In [13]:
print("Cycle WAV path:")
print(cycles_df["wav_path"].iloc[0])

print("\nAudio info file:")
print(audio_info_df["file"].iloc[0])

Cycle WAV path:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/101_1b1_Al_sc_Meditron.wav

Audio info file:
101_1b1_Al_sc_Meditron.wav


In [14]:
print("=== FIXED RESPIRATORY CYCLE TIME VALIDATION ===")

cycles_check = cycles_df.copy()

# Extract only the filename from the full WAV path
cycles_check["file"] = cycles_check["wav_path"].apply(
    lambda x: Path(x).name
)

# Calculate cycle duration
cycles_check["cycle_duration"] = (
    cycles_check["end_time"] - cycles_check["start_time"]
)

# Attach recording duration
cycles_check = cycles_check.merge(
    audio_info_df[["file", "duration"]],
    on="file",
    how="left"
)

print("Invalid start times:",
      (cycles_check["start_time"] < 0).sum())

print("Invalid end times:",
      (cycles_check["end_time"] <= cycles_check["start_time"]).sum())

print("Zero/negative cycle durations:",
      (cycles_check["cycle_duration"] <= 0).sum())

print("Cycles with missing audio duration:",
      cycles_check["duration"].isna().sum())

print(
    "Cycles extending beyond recording:",
    (
        cycles_check["end_time"] >
        cycles_check["duration"]
    ).sum()
)

=== FIXED RESPIRATORY CYCLE TIME VALIDATION ===
Invalid start times: 0
Invalid end times: 0
Zero/negative cycle durations: 0
Cycles with missing audio duration: 0
Cycles extending beyond recording: 0


In [15]:
print("=== RESPIRATORY CYCLE DURATION ANALYSIS ===")

print(
    cycles_check["cycle_duration"].describe()
)

print("\nShortest cycles:")
display(
    cycles_check[
        ["patient_id", "recording_id", "cycle_number", "start_time",
         "end_time", "cycle_duration", "sound_label"]
    ]
    .sort_values("cycle_duration")
    .head(10)
)

print("\nLongest cycles:")
display(
    cycles_check[
        ["patient_id", "recording_id", "cycle_number", "start_time",
         "end_time", "cycle_duration", "sound_label"]
    ]
    .sort_values("cycle_duration", ascending=False)
    .head(10)
)

=== RESPIRATORY CYCLE DURATION ANALYSIS ===
count    6898.000000
mean        2.700509
std         1.172534
min         0.200000
25%         1.932000
50%         2.537000
75%         3.369000
max        16.163000
Name: cycle_duration, dtype: float64

Shortest cycles:


,patient_id,recording_id,cycle_number,start_time,end_time,cycle_duration,sound_label
6522,217,1b1,11,19.764,19.964,0.200,Normal
2464,149,1b1,1,0.022,0.250,0.228,Wheezes
1847,135,2b3,9,19.750,19.993,0.243,Normal
672,117,1b2,7,19.707,19.964,0.257,Normal
2079,140,2b2,11,19.736,19.993,0.257,Normal
2497,150,1b2,17,19.164,19.450,0.286,Normal
2119,141,1b2,6,19.664,19.964,0.300,Normal
5056,188,1b1,1,0.022,0.322,0.300,Normal
1820,135,2b3,9,19.636,19.936,0.300,Normal
894,122,2b2,7,19.679,19.979,0.300,Normal



Longest cycles:


,patient_id,recording_id,cycle_number,start_time,end_time,cycle_duration,sound_label
3667,166,1p1,6,16.7030,32.866,16.1630,Normal
3046,157,1b1,6,36.1240,47.329,11.2050,Normal
3045,157,1b1,5,26.2090,36.124,9.9150,Normal
6737,222,1b1,5,29.9480,39.386,9.4380,Normal
4302,175,1b1,4,12.4630,21.680,9.2170,Wheezes
6564,218,1b1,11,40.0590,48.795,8.7360,Crackles
6769,222,1b1,7,31.4340,40.124,8.6900,Normal
4294,175,1b1,4,9.6323,18.224,8.5917,Both
4290,175,1b1,3,7.7774,16.368,8.5906,Wheezes
4298,175,1b1,4,8.8603,17.264,8.4037,Wheezes


In [16]:
print("=== EXTREME CYCLE DURATION CHECK ===")

print("Cycles shorter than 0.5 seconds:",
      (cycles_check["cycle_duration"] < 0.5).sum())

print("Cycles longer than 10 seconds:",
      (cycles_check["cycle_duration"] > 10).sum())

print("Cycles longer than 15 seconds:",
      (cycles_check["cycle_duration"] > 15).sum())

=== EXTREME CYCLE DURATION CHECK ===
Cycles shorter than 0.5 seconds: 41
Cycles longer than 10 seconds: 2
Cycles longer than 15 seconds: 1


In [17]:
print("=== CYCLE OVERLAP CHECK ===")

overlap_count = 0
overlap_examples = []

for recording_id, group in cycles_check.groupby("file"):

    group = group.sort_values("start_time")

    previous_end = None

    for _, row in group.iterrows():

        if previous_end is not None and row["start_time"] < previous_end:
            overlap_count += 1

            if len(overlap_examples) < 10:
                overlap_examples.append({
                    "file": recording_id,
                    "start_time": row["start_time"],
                    "end_time": row["end_time"],
                    "previous_end": previous_end
                })

        previous_end = max(
            previous_end if previous_end is not None else row["end_time"],
            row["end_time"]
        )

print("Overlapping cycle pairs:", overlap_count)

if overlap_examples:
    print("\nExamples:")
    display(pd.DataFrame(overlap_examples))

=== CYCLE OVERLAP CHECK ===
Overlapping cycle pairs: 0


In [18]:
print("=== AUDIO QUALITY CHECK ===")

print("\nSample-rate distribution:")
print(
    audio_info_df["sample_rate"]
    .value_counts()
    .sort_index()
)

print("\nFrame-count statistics:")
print(
    audio_info_df["frames"].describe()
)

print("\nDuration statistics:")
print(
    audio_info_df["duration"].describe()
)

print("\nInvalid sample rates:",
      (audio_info_df["sample_rate"] <= 0).sum())

print("Invalid frame counts:",
      (audio_info_df["frames"] <= 0).sum())

print("Invalid durations:",
      (audio_info_df["duration"] <= 0).sum())

print("\nRecordings shorter than 1 second:",
      (audio_info_df["duration"] < 1).sum())

=== AUDIO QUALITY CHECK ===

Sample-rate distribution:
sample_rate
4000      90
10000      6
44100    824
Name: count, dtype: int64

Frame-count statistics:
count       920.000000
mean     805551.798913
std      226261.048242
min       31424.000000
25%      882000.000000
50%      882000.000000
75%      882000.000000
max      882001.000000
Name: frames, dtype: float64

Duration statistics:
count    920.000000
mean      21.492424
std        8.307421
min        7.856000
25%       20.000000
50%       20.000000
75%       20.000000
max       86.200000
Name: duration, dtype: float64

Invalid sample rates: 0
Invalid frame counts: 0
Invalid durations: 0

Recordings shorter than 1 second: 0


In [19]:
print("=== RECORDING-CYCLE CONSISTENCY ===")

cycle_counts = (
    cycles_df
    .groupby("wav_path")
    .size()
    .reset_index(name="cycle_count")
)

recording_cycle_check = recordings_df.merge(
    cycle_counts,
    on="wav_path",
    how="left"
)

recording_cycle_check["cycle_count"] = (
    recording_cycle_check["cycle_count"]
    .fillna(0)
)

print("Total recordings:",
      len(recording_cycle_check))

print("Recordings with no cycles:",
      (recording_cycle_check["cycle_count"] == 0).sum())

print("\nCycles per recording:")
print(
    recording_cycle_check["cycle_count"].describe()
)

print("\nRecordings with the fewest cycles:")
display(
    recording_cycle_check
    .sort_values("cycle_count")
    [["patient_id", "recording_id", "wav_path", "cycle_count"]]
    .head(10)
)

=== RECORDING-CYCLE CONSISTENCY ===
Total recordings: 920
Recordings with no cycles: 0

Cycles per recording:
count    920.000000
mean       7.497826
std        3.335762
min        2.000000
25%        5.000000
50%        7.000000
75%        9.000000
max       33.000000
Name: cycle_count, dtype: float64

Recordings with the fewest cycles:


,patient_id,recording_id,wav_path,cycle_count
417,160,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2
416,160,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2
415,160,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2
414,160,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2
413,160,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2
412,160,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2
437,162,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3
438,162,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3
439,162,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3
440,162,1b2,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3


In [20]:
import wave

print("=== WAV FILE READABILITY CHECK ===")

unreadable_files = []

for wav_path in recordings_df["wav_path"]:
    try:
        with wave.open(wav_path, "rb") as audio:
            frames = audio.getnframes()
            sample_rate = audio.getframerate()

            if frames <= 0 or sample_rate <= 0:
                unreadable_files.append(
                    (wav_path, "Invalid audio properties")
                )

    except Exception as e:
        unreadable_files.append(
            (wav_path, str(e))
        )

print("Total WAV files checked:", len(recordings_df))
print("Unreadable/invalid WAV files:", len(unreadable_files))

if unreadable_files:
    print("\nExamples:")
    display(
        pd.DataFrame(
            unreadable_files[:10],
            columns=["wav_path", "error"]
        )
    )
    

=== WAV FILE READABILITY CHECK ===
Total WAV files checked: 920
Unreadable/invalid WAV files: 0


In [21]:
import hashlib

print("=== AUDIO CONTENT DUPLICATE CHECK ===")

def file_md5(file_path):
    md5 = hashlib.md5()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()


recording_hashes = []

for wav_path in recordings_df["wav_path"]:
    recording_hashes.append(file_md5(wav_path))

hash_df = recordings_df.copy()
hash_df["audio_md5"] = recording_hashes

duplicate_hash_count = (
    hash_df["audio_md5"].duplicated(keep=False).sum()
)

unique_hash_count = hash_df["audio_md5"].nunique()

print("Total WAV files:", len(hash_df))
print("Unique audio hashes:", unique_hash_count)
print("Files belonging to duplicate hash groups:",
      duplicate_hash_count)

=== AUDIO CONTENT DUPLICATE CHECK ===
Total WAV files: 920
Unique audio hashes: 920
Files belonging to duplicate hash groups: 0


In [22]:
print("=== PATIENT → RECORDING → CYCLE HIERARCHY CHECK ===")

patient_recording_counts = (
    recordings_df
    .groupby("patient_id")
    .size()
)

patient_cycle_counts = (
    cycles_df
    .groupby("patient_id")
    .size()
)

print("Patients with recordings:",
      patient_recording_counts.index.nunique())

print("Patients with cycles:",
      patient_cycle_counts.index.nunique())

print(
    "Patients with recordings but no cycles:",
    len(
        set(patient_recording_counts.index)
        - set(patient_cycle_counts.index)
    )
)

print(
    "Patients with cycles but no recordings:",
    len(
        set(patient_cycle_counts.index)
        - set(patient_recording_counts.index)
    )
)

print("\nRecordings per patient:")
print(patient_recording_counts.describe())

print("\nCycles per patient:")
print(patient_cycle_counts.describe())

=== PATIENT → RECORDING → CYCLE HIERARCHY CHECK ===
Patients with recordings: 126
Patients with cycles: 126
Patients with recordings but no cycles: 0
Patients with cycles but no recordings: 0

Recordings per patient:
count    126.000000
mean       7.301587
std        9.519050
min        1.000000
25%        1.000000
50%        3.000000
75%        8.000000
max       66.000000
dtype: float64

Cycles per patient:
count    126.000000
mean      54.746032
std       67.686328
min        4.000000
25%       11.000000
50%       26.500000
75%       74.000000
max      507.000000
dtype: float64


In [23]:
cleaning_summary = {
    "patients": len(patients_df),
    "recordings": len(recordings_df),
    "respiratory_cycles": len(cycles_df),

    "duplicate_patients": patients_df.duplicated(
        subset=["patient_id"]
    ).sum(),

    "duplicate_recordings": recordings_df.duplicated(
        subset=["wav_path"]
    ).sum(),

    "duplicate_cycles": cycles_df.duplicated(
        subset=[
            "patient_id",
            "recording_id",
            "chest_location",
            "acquisition_mode",
            "equipment",
            "cycle_number"
        ]
    ).sum(),

    "unreadable_wav_files": len(unreadable_files),

    "unique_audio_hashes": hash_df["audio_md5"].nunique(),

    "recordings_without_cycles": (
        recording_cycle_check["cycle_count"] == 0
    ).sum(),

    "cycles_with_invalid_timing": (
        (cycles_check["start_time"] < 0) |
        (cycles_check["end_time"] <= cycles_check["start_time"]) |
        (cycles_check["end_time"] > cycles_check["duration"])
    ).sum(),

    "overlapping_cycles": overlap_count,
}

cleaning_summary


{'patients': 126,
 'recordings': 920,
 'respiratory_cycles': 6898,
 'duplicate_patients': np.int64(0),
 'duplicate_recordings': np.int64(0),
 'duplicate_cycles': np.int64(0),
 'unreadable_wav_files': 0,
 'unique_audio_hashes': 920,
 'recordings_without_cycles': np.int64(0),
 'cycles_with_invalid_timing': np.int64(0),
 'overlapping_cycles': 0}

In [24]:
# ============================================
# SAVE CLEANED PROCESSED TABLES
# ============================================

print("=== SAVING CLEANED DATA ===")

# Create independent copies
patients_clean = patients_df.copy()
recordings_clean = recordings_df.copy()
cycles_clean = cycles_df.copy()
audio_info_clean = audio_info_df.copy()

# Save cleaned tables
patients_clean.to_csv(
    PROCESSED_DATA_DIR / "patients_clean.csv",
    index=False
)

recordings_clean.to_csv(
    PROCESSED_DATA_DIR / "recordings_clean.csv",
    index=False
)

cycles_clean.to_csv(
    PROCESSED_DATA_DIR / "cycles_clean.csv",
    index=False
)

audio_info_clean.to_csv(
    PROCESSED_DATA_DIR / "audio_info_clean.csv",
    index=False
)

# Save cleaning summary
cleaning_summary_df = pd.DataFrame(
    list(cleaning_summary.items()),
    columns=["check", "result"]
)

cleaning_summary_df.to_csv(
    PROCESSED_DATA_DIR / "cleaning_summary.csv",
    index=False
)

print("\nSaved files:")

for file in sorted(PROCESSED_DATA_DIR.glob("*.csv")):
    print("✓", file.name)

=== SAVING CLEANED DATA ===

Saved files:
✓ audio_info_clean.csv
✓ cleaning_summary.csv
✓ cycles_clean.csv
✓ patients_clean.csv
✓ recordings_clean.csv


In [25]:
# ============================================
# VERIFY SAVED PROCESSED TABLES
# ============================================

print("=== VERIFYING PROCESSED DATA ===")

patients_verify = pd.read_csv(
    PROCESSED_DATA_DIR / "patients_clean.csv"
)

recordings_verify = pd.read_csv(
    PROCESSED_DATA_DIR / "recordings_clean.csv"
)

cycles_verify = pd.read_csv(
    PROCESSED_DATA_DIR / "cycles_clean.csv"
)

audio_info_verify = pd.read_csv(
    PROCESSED_DATA_DIR / "audio_info_clean.csv"
)

summary_verify = pd.read_csv(
    PROCESSED_DATA_DIR / "cleaning_summary.csv"
)

print("\nRows after saving:")
print("Patients:", len(patients_verify))
print("Recordings:", len(recordings_verify))
print("Cycles:", len(cycles_verify))
print("Audio info:", len(audio_info_verify))

print("\nExpected:")
print("Patients: 126")
print("Recordings: 920")
print("Cycles: 6898")
print("Audio info: 920")

print("\nCleaning summary:")
display(summary_verify)

=== VERIFYING PROCESSED DATA ===

Rows after saving:
Patients: 126
Recordings: 920
Cycles: 6898
Audio info: 920

Expected:
Patients: 126
Recordings: 920
Cycles: 6898
Audio info: 920

Cleaning summary:


,check,result
0,patients,126
1,recordings,920
2,respiratory_cycles,6898
3,duplicate_patients,0
4,duplicate_recordings,0
5,duplicate_cycles,0
6,unreadable_wav_files,0
7,unique_audio_hashes,920
8,recordings_without_cycles,0
9,cycles_with_invalid_timing,0


# Final Data Cleaning Summary

## Dataset After Cleaning and Validation

The ICBHI respiratory sound dataset was systematically checked at the patient, recording, audio, and respiratory-cycle levels.

### Dataset Size

- Patients: **126**
- Recordings: **920**
- Respiratory cycles: **6,898**
- Audio metadata records: **920**

### Patient-Level Validation

- No duplicate patients were found.
- Diagnosis information is available for all 126 patients.
- Decimal age values were preserved as provided in the dataset.
- Missing demographic values were investigated according to age group.
- Missing adult BMI for children and missing child-specific measurements for adults were treated as expected structural missingness.
- Patient 223 has missing demographic information but has a valid COPD diagnosis, 6 recordings, and 61 respiratory cycles. Therefore, the patient was retained.
- No demographic values were artificially imputed during cleaning.

### Recording-Level Validation

- All 920 recordings have valid ICBHI filename metadata.
- All recordings have valid chest-location, acquisition-mode, and equipment values.
- The official recording-level split contains 539 training recordings and 381 test recordings.
- Every recording has at least one annotated respiratory cycle.
- No duplicate recording paths were found.

### Audio Validation

- All 920 WAV files were successfully opened and read.
- No invalid sample rates, frame counts, or durations were found.
- The dataset contains recordings with different original sampling rates; these were preserved during cleaning.
- No recording was shorter than one second.
- MD5 hashing found 920 unique audio files, with no byte-identical duplicates.
- Audio resampling and other signal-processing operations were intentionally not performed during this cleaning stage and will be handled during audio preprocessing.

### Respiratory-Cycle Validation

- All 6,898 respiratory cycles have valid start and end times.
- No cycle has a zero or negative duration.
- No cycle extends beyond its corresponding recording.
- No overlapping respiratory-cycle annotations were found.
- Crackle and wheeze annotations contain only valid binary values.
- The four expected sound classes are present:
  - Normal
  - Crackles
  - Wheezes
  - Both
- No respiratory cycles were removed solely because of unusual duration.

### Cleaning Decision

The validation process did not identify corrupted, duplicated, structurally invalid, or temporally invalid data requiring removal.

Therefore, **all 126 patients, 920 recordings, and 6,898 respiratory cycles were retained**.

The validated tables were saved in:

`data/processed/`

as:

- `patients_clean.csv`
- `recordings_clean.csv`
- `cycles_clean.csv`
- `audio_info_clean.csv`
- `cleaning_summary.csv`

The original raw ICBHI audio and annotation files were not modified.